# Student Interface

Submit a Python solution and track its grading:
1. **Your details** -- name + email (used to identify your submissions).
2. **Browse tasks** -- see published tasks and their ids.
3. **Submit a solution** -- upload a `.py` file from your browser, or point at one already on this machine.
4. **Check status & grades** -- see all your submissions, or watch one until it's graded.

Run the cells top to bottom once; each section's controls can be reused afterwards.

In [ ]:
import time

import httpx
import ipywidgets as widgets
from IPython.display import display, Markdown

# Base URL of the running api.py (see Grader/Dockerfile / docker-compose.yml).
API_BASE_URL = "http://localhost:8000"

client = httpx.Client(base_url=API_BASE_URL, timeout=30.0)

print(f"Talking to {API_BASE_URL}")

In [ ]:
import html as _html
import uuid as _uuid

def render_table(rows, columns, sortable=True):
    """Renders rows as an HTML table. If sortable, clicking a header sorts by
    that column (numeric if every cell in it parses as a number, else text),
    toggling ascending/descending on repeat clicks -- plain vanilla JS, no
    library, scoped to this table's own id so multiple tables on the page
    don't interfere with each other."""
    if not rows:
        return widgets.HTML("<i>No rows.</i>")

    table_id = f"tbl_{_uuid.uuid4().hex}"

    def header_cell(i, c):
        label = _html.escape(str(c))
        if not sortable:
            return f"<th style='text-align:left; padding:4px 10px'>{label}</th>"
        return (
            f"<th onclick=\"_sortTable('{table_id}', {i})\" "
            f"style='text-align:left; padding:4px 10px; cursor:pointer; user-select:none' "
            f"title='Click to sort'>{label} \u21c5</th>"
        )

    head = "".join(header_cell(i, c) for i, c in enumerate(columns))
    body = "".join(
        "<tr>" + "".join(
            f"<td style='padding:4px 10px'>{_html.escape(str(row.get(c, '')))}</td>" for c in columns
        ) + "</tr>"
        for row in rows
    )
    script = "" if not sortable else """
<script>
window._sortTable = window._sortTable || function(tableId, colIdx) {
    var table = document.getElementById(tableId);
    var tbody = table.tBodies[0];
    var rows = Array.prototype.slice.call(tbody.rows);
    var asc = !(table.getAttribute('data-sort-col') == colIdx && table.getAttribute('data-sort-dir') === 'asc');
    function cellText(row) { return row.cells[colIdx].innerText.trim(); }
    var allNumeric = rows.every(function(r) { var t = cellText(r); return t === '' || !isNaN(parseFloat(t)); });
    rows.sort(function(a, b) {
        var ta = cellText(a), tb = cellText(b), cmp;
        cmp = allNumeric ? ((parseFloat(ta) || 0) - (parseFloat(tb) || 0)) : ta.localeCompare(tb);
        return asc ? cmp : -cmp;
    });
    rows.forEach(function(r) { tbody.appendChild(r); });
    table.setAttribute('data-sort-col', colIdx);
    table.setAttribute('data-sort-dir', asc ? 'asc' : 'desc');
};
</script>
"""
    return widgets.HTML(f"<table id='{table_id}' style='border-collapse:collapse'><tr>{head}</tr>{body}</table>{script}")


def api_error_detail(response):
    try:
        return response.json().get("detail", response.text)
    except Exception:
        return response.text


def api_request(method, path, **kwargs):
    """client.request, but a connection failure prints a message instead of
    raising -- so a refresh/submit button click never dies with a raw
    traceback just because the API isn't reachable yet."""
    try:
        return client.request(method, path, **kwargs)
    except httpx.RequestError as e:
        print(f"Could not reach {API_BASE_URL}{path}: {e}")
        return None

## Part 1: Your details

In [ ]:
student_name_box = widgets.Text(description="Full name:", style={"description_width": "100px"}, layout=widgets.Layout(width="400px"))
student_email_box = widgets.Text(description="Email:", style={"description_width": "100px"}, layout=widgets.Layout(width="400px"))

display(widgets.VBox([student_name_box, student_email_box]))

## Part 2: Browse tasks
Published tasks you can submit solutions for. Copy a task's id into Part 3 below.

In [ ]:
tasks_out = widgets.Output()
refresh_tasks_btn = widgets.Button(description="Refresh tasks")

def on_refresh_tasks(_):
    with tasks_out:
        tasks_out.clear_output()
        resp = api_request("GET", "/tasks")
        if resp is None:
            return
        if resp.status_code != 200:
            print(f"Error {resp.status_code}: {api_error_detail(resp)}")
            return
        display(render_table(resp.json(), ["id", "title", "topic", "level"]))

refresh_tasks_btn.on_click(on_refresh_tasks)
on_refresh_tasks(None)

display(widgets.VBox([refresh_tasks_btn, tasks_out]))

## Part 3: Submit a solution
Provide your `.py` file either way -- whichever one has something in it is used (the browser upload takes priority if you fill in both):
- **Upload from browser** -- opens your browser's file picker.
- **Path on this machine** -- a file already sitting somewhere Jupyter can read, e.g. `solution.py` or `/home/me/task3/solution.py`.

After submitting, this watches the submission until it's graded (or until the wait times out -- grading runs in the background, so you can also just check on it later in Part 4).

In [ ]:
submit_task_id = widgets.Text(description="Task id:", style={"description_width": "100px"}, layout=widgets.Layout(width="500px"))
submit_upload = widgets.FileUpload(description="Upload .py", accept=".py", multiple=False)
submit_local_path = widgets.Text(description="...or path:", style={"description_width": "100px"}, layout=widgets.Layout(width="500px"))
submit_btn = widgets.Button(description="Submit solution", button_style="success")
submit_out = widgets.Output()

WATCH_INTERVAL_SECONDS = 2
WATCH_TIMEOUT_SECONDS = 120

def _load_solution_bytes():
    """Returns (filename, content_bytes), preferring the browser upload."""
    if submit_upload.value:
        uploaded = submit_upload.value[0] if isinstance(submit_upload.value, tuple) else list(submit_upload.value.values())[0]
        return uploaded["name"], bytes(uploaded["content"])

    path = submit_local_path.value.strip()
    if path:
        with open(path, "rb") as f:
            return path.split("/")[-1], f.read()

    return None, None


def render_submission_status(data):
    lines = [
        f"**Submission `{data['id']}`** -- status: **{data['status']}**",
        f"- Task: `{data['task_id']}`",
    ]
    if data["tests_passed"] is not None:
        lines.append(f"- Auto-graded: {data['tests_passed']} / {data['tests_total']} test case(s) passed")
    if data.get("feedback"):
        lines.append("\n**Feedback:**\n> " + data["feedback"].replace("\n", "\n> "))
    if data.get("final_score") is not None:
        lines.append(f"\n**Final score:** {data['final_score']}")
    elif data.get("human_score") is not None:
        lines.append(f"\n**Instructor score:** {data['human_score']}")
    if data.get("human_feedback"):
        lines.append("\n**Instructor feedback:**\n> " + data["human_feedback"].replace("\n", "\n> "))
    return Markdown("\n".join(lines))


def on_submit(_):
    with submit_out:
        submit_out.clear_output()

        task_id = submit_task_id.value.strip()
        name = student_name_box.value.strip()
        email = student_email_box.value.strip()
        if not task_id or not name or not email:
            print("Fill in your name/email (Part 1) and a task id.")
            return

        try:
            filename, content = _load_solution_bytes()
        except OSError as e:
            print(f"Could not read local file: {e}")
            return
        if content is None:
            print("Upload a .py file or enter a local path.")
            return

        resp = api_request(
            "POST",
            "/submissions",
            data={"task_id": task_id, "student_email": email, "student_full_name": name},
            files={"file": (filename, content, "text/x-python")},
        )
        if resp is None:
            return
        if resp.status_code != 202:
            print(f"Error {resp.status_code}: {api_error_detail(resp)}")
            return

        submission_id = resp.json()["submission_id"]
        print(f"Submitted as {submission_id}. Watching for a grade...\n")

        deadline = time.time() + WATCH_TIMEOUT_SECONDS
        last = None
        while time.time() < deadline:
            r = api_request("GET", f"/submissions/{submission_id}")
            if r is None:
                return
            if r.status_code != 200:
                print(f"Error {r.status_code}: {api_error_detail(r)}")
                return
            last = r.json()
            if last["status"] not in ("submitted", "checking"):
                break
            time.sleep(WATCH_INTERVAL_SECONDS)

        display(render_submission_status(last))
        if last["status"] in ("submitted", "checking"):
            print("\nStill grading -- check back later in Part 4 with this submission id.")

submit_btn.on_click(on_submit)
display(widgets.VBox([submit_task_id, submit_upload, submit_local_path, submit_btn, submit_out]))

## Part 4: Check status & grades
**All my submissions** -- everything you've submitted, using the email from Part 1.
**One submission** -- look up a single submission id (e.g. one from Part 3) and optionally watch it until it's graded.

In [ ]:
my_submissions_out = widgets.Output()
refresh_my_submissions_btn = widgets.Button(description="Refresh my submissions")

def on_refresh_my_submissions(_):
    with my_submissions_out:
        my_submissions_out.clear_output()
        email = student_email_box.value.strip()
        if not email:
            print("Enter your email in Part 1.")
            return
        resp = api_request("GET", f"/students/{email}/submissions")
        if resp is None:
            return
        if resp.status_code != 200:
            print(f"Error {resp.status_code}: {api_error_detail(resp)}")
            return
        rows = resp.json()
        for r in rows:
            r["score"] = "" if r["tests_passed"] is None else f"{r['tests_passed']}/{r['tests_total']}"
            if r["final_score"] is None:
                r["final_score"] = ""
        display(render_table(rows, ["submitted_at", "task_title", "status", "score", "final_score", "id"]))

refresh_my_submissions_btn.on_click(on_refresh_my_submissions)
display(widgets.VBox([refresh_my_submissions_btn, my_submissions_out]))

In [ ]:
lookup_submission_id = widgets.Text(description="Submission id:", style={"description_width": "120px"}, layout=widgets.Layout(width="500px"))
lookup_btn = widgets.Button(description="Check now")
watch_btn = widgets.Button(description="Watch until graded", button_style="info")
lookup_out = widgets.Output()

def _fetch_and_render(submission_id):
    resp = api_request("GET", f"/submissions/{submission_id}")
    if resp is None:
        return None
    if resp.status_code != 200:
        print(f"Error {resp.status_code}: {api_error_detail(resp)}")
        return None
    data = resp.json()
    display(render_submission_status(data))
    return data

def on_lookup(_):
    with lookup_out:
        lookup_out.clear_output()
        sub_id = lookup_submission_id.value.strip()
        if not sub_id:
            print("Enter a submission id.")
            return
        _fetch_and_render(sub_id)

def on_watch(_):
    with lookup_out:
        lookup_out.clear_output()
        sub_id = lookup_submission_id.value.strip()
        if not sub_id:
            print("Enter a submission id.")
            return
        deadline = time.time() + WATCH_TIMEOUT_SECONDS
        while time.time() < deadline:
            data = _fetch_and_render(sub_id)
            if data is None or data["status"] not in ("submitted", "checking"):
                return
            time.sleep(WATCH_INTERVAL_SECONDS)
            lookup_out.clear_output()
        print("Timed out waiting -- still grading, try again later.")

lookup_btn.on_click(on_lookup)
watch_btn.on_click(on_watch)
display(widgets.VBox([lookup_submission_id, widgets.HBox([lookup_btn, watch_btn]), lookup_out]))